In [8]:
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt


# ===============================
# Step 1: 读取视频16帧
# ===============================

def load_clip(txt_path, resize=(224, 224)):
    """
    读取txt中列出的帧路径，返回 numpy 数组 [T,H,W,C]
    """
    with open(txt_path, 'r') as f:
        paths = [l.strip() for l in f.readlines() if l.strip()]

    frames = []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            raise FileNotFoundError(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if resize:
            img = cv2.resize(img, resize)
        frames.append(img.astype(np.float32) / 255.0)

    return np.stack(frames, axis=0)  # (T,H,W,C)


# ===============================
# Step 2: 频域分析 + 空间掩膜
# ===============================

def frequency_guided_mask(
    video_clip,
    freq_top_ratio=0.1,
    spatial_mask_ratio=0.1
):
    """
    从频域挑选高能量分量，逆变换后在空间域mask高响应区域
    freq_top_ratio:   保留最高能量频率比例
    spatial_mask_ratio: 在像素域mask掉高响应区域比例
    """
    T, H, W, C = video_clip.shape

    # --- 3D FFT并计算能量 ---
    fft_clip = np.fft.fftn(video_clip, axes=(0, 1, 2))
    fft_amp = np.abs(fft_clip)
    energy = np.mean(fft_amp, axis=-1)  # (T,H,W)

    # --- 挑出Top频率 ---
    flatE = energy.flatten()
    threshold = np.quantile(flatE, 1 - freq_top_ratio)
    high_freq_mask = (energy >= threshold).astype(np.float32)

    # --- 反变换到空间域获取显著性图 ---
    fft_masked = fft_clip * high_freq_mask[..., None]
    spatial_heat = np.abs(np.fft.ifftn(fft_masked, axes=(0, 1, 2)))
    heatmap = np.mean(spatial_heat, axis=0)  # (H,W)
    heatmap = heatmap / (heatmap.max() + 1e-6)

    # --- 空间域选择最显著区域mask ---
    spatial_thresh = np.quantile(heatmap, 1 - spatial_mask_ratio)
    pixel_mask = (heatmap >= spatial_thresh).astype(np.float32)  # (H,W)

    # --- 在像素域施加mask ---
    pixel_mask_3d = pixel_mask[..., None]  # (H,W,1)
    print("video_clip:", video_clip.shape)
    print("pixel_mask_3d:", pixel_mask_3d.shape)

    masked_clip = video_clip * (1 - pixel_mask_3d)  # 广播到 (T,H,W,C)
    return masked_clip, heatmap, pixel_mask, high_freq_mask


# ===============================
# Step 3: 可视化/保存结果
# ===============================

def visualize_frequency_and_spatial_maps(
    video_clip,
    masked_clip,
    heatmap,
    pixel_mask,
    out_dir="freq_guided_vis"
):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    T = video_clip.shape[0]

    # 生成彩色热图
    heatmap_color = (plt.cm.jet(heatmap)[:, :, :3] * 255).astype(np.uint8)
    mask_binary = (pixel_mask * 255).astype(np.uint8)

    for i in range(T):
        original = (video_clip[i] * 255).astype(np.uint8)
        overlay = cv2.addWeighted(original, 0.6, heatmap_color, 0.4, 0)

        # 红色边框显示mask区域
        overlay_masked = overlay.copy()
        contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay_masked, contours, -1, (255, 0, 0), 2)

        cv2.imwrite(f"{out_dir}/overlay_heat_mask_{i+1:03d}.jpg",
                    cv2.cvtColor(overlay_masked, cv2.COLOR_RGB2BGR))

    # 全局热图保存
    plt.figure(figsize=(5, 5))
    plt.imshow(heatmap, cmap="jet")
    plt.colorbar(label="Frequency-guided spatial saliency")
    plt.title("Spatial frequency influence heatmap")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/heatmap_global.png", dpi=150)
    plt.close()

    # 差分图
    diff = np.mean(np.abs(video_clip - masked_clip), axis=-1)
    diff = diff / (diff.max() + 1e-6)
    for i in range(T):
        diff_img = (diff[i] * 255).astype(np.uint8)
        cv2.imwrite(f"{out_dir}/diff_{i+1:03d}.jpg", diff_img)

    print(f"[✓] Saved results in: {out_dir}/")


# ===============================
# Step 4: 主运行流程
# ===============================

def main():
    txt_path = (
        "data/Surge_Frames/CATARACTS/clips_16f/"
        "clip_dense_16f_info/train/case1_c014_f-00001-000015_padded.txt"
    )

    print("[*] Loading frames...")
    video = load_clip(txt_path)

    print("[*] Computing frequency-guided spatial masking...")
    masked_clip, heatmap, pixel_mask, high_freq_mask = frequency_guided_mask(
        video,
        freq_top_ratio=0.1,
        spatial_mask_ratio=0.1
    )

    print("[*] Visualizing...")
    visualize_frequency_and_spatial_maps(video, masked_clip, heatmap, pixel_mask)

    print("\n✅ Done! Check folder: freq_guided_vis/\n")


if __name__ == "__main__":
    main()

[*] Loading frames...
[*] Computing frequency-guided spatial masking...
video_clip: (16, 224, 224, 3)
pixel_mask_3d: (224, 224, 3, 1)


ValueError: operands could not be broadcast together with shapes (16,224,224,3) (224,224,3,1) 